In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

# Initialize the Model
# We use a low temperature (0) because classification requires deterministic logic, not creativity.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [2]:
# Define the desired data structure
class TicketInfo(BaseModel):
    category: str = Field(
        description="The category of the ticket. Options: ['Refund', 'Technical Issue', 'General Inquiry', 'Spam']"
    )
    
    urgency: str = Field(
        description="The urgency based on sentiment. Options: ['Low', 'Medium', 'High']"
    )
    
    summary: str = Field(
        description="A concise 1-sentence summary of the issue"
    )
    
    order_id: str = Field(
        description="The Order ID if present, otherwise 'Unknown'"
    )


# Create the parser
parser = JsonOutputParser(pydantic_object=TicketInfo)

In [3]:
# Define the Prompt
system_template = """
You are a Senior Customer Support Manager.
Your job is to analyze incoming emails and route them correctly.
Extract the relevant information and format it exactly as requested.

{format_instructions}
"""

human_template = """
Incoming Email:
{email_content}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", human_template)
])


# Inject the parser's instructions into the prompt
# This adds text like: "The output should be a markdown code snippet formatted in the following schema: ..."
prompt = prompt.partial(
    format_instructions=parser.get_format_instructions()
)

In [4]:
# create the chain
chain = prompt | llm | parser

In [5]:
# Test Case 1: An angry customer with a technical issue
email_1 = """
Subject: IT'S BROKEN AGAIN!!!
Hi, I received my order #998877 yesterday. I turned it on and the screen just flashed and died.
I need this fixed immediately or I want my money back. This is unacceptable!
"""


# Test Case 2: A polite inquiry
email_2 = """
Subject: Question about shipping
Hello, I was just wondering if you ship to Canada? I couldn't find it on your FAQ.
Thanks, Sarah.
"""


print("--- Processing Email 1 ---")
result_1 = chain.invoke({"email_content": email_1})
print(result_1)


print("\n--- Processing Email 2 ---")
result_2 = chain.invoke({"email_content": email_2})
print(result_2)

--- Processing Email 1 ---
{'category': 'Technical Issue', 'urgency': 'High', 'summary': 'The screen of the order #998877 flashed and died upon turning it on.', 'order_id': '998877'}

--- Processing Email 2 ---
{'category': 'General Inquiry', 'urgency': 'Low', 'summary': 'Customer inquiring about shipping to Canada.', 'order_id': 'Unknown'}


In [6]:
# Test Case 3: A customer requesting a refund
email_3 = """
Subject: Refund request for my laptop

Hello, I would like to return my laptop and request a refund.
My order number is #445566. The laptop is still unopened and the original seal is intact.
Please let me know how I can proceed with the return.

Thanks,
Michael
"""

print("\n--- Processing Email 3 ---")
result_3 = chain.invoke({"email_content": email_3})
print(result_3)


--- Processing Email 3 ---
{'category': 'Refund', 'urgency': 'Medium', 'summary': 'Request for a refund for an unopened laptop.', 'order_id': '445566'}
